# 02 — Implied Volatility Surface Construction

In BSM world, implied vol is flat across strikes and expiries — a single number. 
In reality, it is not. The market prices different strikes at different vols, reflecting:

- **Crash risk / left tail**: put buyers pay a premium for downside protection → put wing is elevated (negative skew)
- **Supply/demand imbalances**: structural demand for OTM puts from portfolio managers
- **Jumps**: BSM cannot price jump risk; the smile is partly compensation for this

This notebook constructs the full implied vol surface for SPY:
1. Fetch live options chain
2. Extract implied vols
3. Fit a smooth surface
4. Visualise: 3D surface and smile slices

> **What to watch for:** the surface is noisy near the wings (wide bid-ask, low OI) and at very short expiries.
> Our filters handle this, but some residual noise is expected and acknowledged.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.surface import VolSurface

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Fetch & Clean Options Data

We use yfinance to pull the SPY options chain. Filters applied:
- Expiry window: 7–365 DTE
- Minimum open interest: 100 contracts
- Minimum volume: 10 contracts
- Moneyness range: 60%–140% of spot (removes illiquid deep wings)
- Bid > 0 (removes stale/zero-bid quotes)

In [ ]:
surf = VolSurface(
    ticker="SPY",
    q=0.013,          # approximate SPY dividend yield
    min_dte=7,
    max_dte=365,
    min_open_interest=100,
    min_volume=10,
)

print(f"Spot: {surf.spot:.2f}")
print(f"Risk-free rate: {surf.r:.3%}")
print(f"Valid options with IV: {len(surf.raw)}")
surf.raw.head()

## 2. IV Distribution

Before fitting the surface, inspect the raw extracted IVs.
Suspicious values (very high or very low) indicate stale quotes or data issues.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# IV histogram
axes[0].hist(surf.raw["iv"] * 100, bins=50, color="#2196F3", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("Implied Vol (%)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Extracted IVs")

# IV scatter by log-moneyness and tenor
sc = axes[1].scatter(
    surf.raw["log_moneyness"],
    surf.raw["T"] * 365,
    c=surf.raw["iv"] * 100,
    cmap="RdYlGn_r",
    s=8, alpha=0.7
)
plt.colorbar(sc, ax=axes[1], label="IV (%)")
axes[1].set_xlabel("Log-Moneyness log(K/F)")
axes[1].set_ylabel("DTE")
axes[1].set_title("Raw IV Scatter — Colour = Vol Level")

plt.tight_layout()
plt.savefig("../figures/02_iv_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Fit the Surface

We fit a 2D cubic spline in (log-moneyness, tenor) space.
Working in log-moneyness rather than absolute strike gives better numerical conditioning:
the smile is roughly symmetric around m=0 (ATM) for most underlyings.

In [ ]:
surf.fit(n_moneyness=35, n_tenors=18)

# Spot-check: ATM 3M vol
atm_3m_iv = surf.iv(surf.spot, 90/365)
print(f"ATM 3M implied vol: {atm_3m_iv:.2%}")

## 4. Vol Surface — 3D Plot

In [ ]:
surf.plot_surface(save_path="../figures/02_vol_surface_3d.png")

## 5. Vol Smiles Across Tenors

The smile slice at each tenor reveals the skew structure.
Key features to note:
- **Steeper skew at short tenors**: short-dated puts are relatively more expensive (near-term tail risk)
- **Flatter smile at long tenors**: mean-reversion dampens the wings over time
- **Put wing elevation**: negative skew is persistent, reflecting structural demand for downside protection

In [ ]:
surf.plot_smiles(
    tenors_days=[30, 60, 90, 180],
    save_path="../figures/02_vol_smiles.png"
)

## 6. Term Structure of ATM Volatility

The ATM term structure — how vol changes with expiry at-the-money — is a key macro signal.
An **inverted** term structure (short-end > long-end) typically reflects elevated near-term uncertainty.

In [ ]:
tenors_days = np.arange(10, 365, 5)
atm_ivs = [surf.iv(surf.spot, t / 365) for t in tenors_days]

plt.figure(figsize=(10, 4))
plt.plot(tenors_days, np.array(atm_ivs) * 100, color="#2196F3", lw=2)
plt.xlabel("Days to Expiry")
plt.ylabel("ATM Implied Vol (%)")
plt.title(f"SPY ATM Volatility Term Structure")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../figures/02_atm_term_structure.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- Real implied vols are not flat: the smile reflects crash risk, jump risk, and supply/demand.
- The negative skew is the dominant feature for equity indices: put wings are structurally elevated.
- The term structure of ATM vol carries macro information: inversions signal near-term stress.

**Next:** Notebook 03 derives the Dupire local vol surface from this fitted implied vol surface.